# The Impact of Keyword-Based Weak Supervision on Technology-Assisted Review Performance


### Final design
1. Clean and deduplicate the Enron corpus.
2. Create a fixed 80/20 development/holdout split.
3. Define and freeze a category-based organizational keyword vocabulary.
4. Apply **whole-word / complete-phrase matching** to create weak labels on development data.
5. Train Logistic Regression, Linear SVM, TextCNN, and DistilBERT on those weak labels.
6. Randomly sample approximately 1,000 emails from the untouched holdout pool and manually label them using a predefined relevance criterion.
7. Evaluate the **raw keyword rule itself** as the baseline and compare it with all four trained models.
8. Report weak-label coverage, keyword frequencies, human-test class distribution, confusion matrices, and bootstrap 95% confidence intervals.


The final human test set must not be used to select keywords, tune the keyword rule, choose model hyperparameters, or make other development decisions.

## Relevance definition used for manual annotation

An email is **Relevant (1)** if its substantive content concerns the day-to-day business, administrative, financial, legal, managerial, personnel, contractual, energy/trading, compliance, planning, or other operational activities of the organization.

An email is **Non-Relevant (0)** if its substantive content is primarily personal/friendly communication, casual conversation unrelated to organizational activity, unsolicited advertising/spam, personal arrangements, or other content unrelated to organizational operations.

Judge the meaning and context of the email rather than whether a particular keyword appears.

This operational definition is specific to this experiment and is not intended to represent responsiveness to every litigation request or legal matter.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install -U transformers datasets accelerate rank_bm25

import os, re, json, random, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = "/content/drive/MyDrive/enron/"
SPLIT_DIR = os.path.join(BASE_DIR, "locked_splits")
ANNOTATION_DIR = os.path.join(BASE_DIR, "annotation")
OUT_DIR = os.path.join(BASE_DIR, "outputs", datetime.now().strftime("%Y%m%d_%H%M%S"))

for p in [SPLIT_DIR, ANNOTATION_DIR, OUT_DIR]:
    os.makedirs(p, exist_ok=True)

print("Output directory:", OUT_DIR)

## 1. Configuration

In [ ]:
CSV_PATH = "/content/drive/MyDrive/enron/emails.csv"

FILE_COL = "file"
RAW_MESSAGE_COL = "message"

HOLDOUT_FRACTION = 0.20
N_HUMAN_TEST = 1000

# Leave False until manual annotation is complete and all development choices are frozen.
FINAL_EVALUATION = True

# None = full development training set.
# For a quick Colab test only, you can temporarily use e.g. 30000.
MAX_TRAIN_ROWS = None

## 2. Load, normalize, deduplicate, and create stable document IDs

In [ ]:
import re
import hashlib
import pandas as pd
from email import policy
from email.parser import Parser

# =========================================================
# 1. LOAD RAW ENRON DATASET
# =========================================================

df = pd.read_csv(CSV_PATH)

assert "file" in df.columns
assert "message" in df.columns

print("Original rows:", len(df))


# =========================================================
# 2. PARSE RAW ENRON EMAILS
# =========================================================

def parse_enron_email(raw_message):
    if pd.isna(raw_message):
        raw_message = ""

    raw_message = str(raw_message)

    try:
        msg = Parser(policy=policy.default).parsestr(raw_message)

        message_id = str(msg.get("Message-ID", "") or "").strip()
        date = str(msg.get("Date", "") or "").strip()
        sender = str(msg.get("From", "") or "").strip()
        recipient = str(msg.get("To", "") or "").strip()
        cc = str(msg.get("Cc", "") or "").strip()
        bcc = str(msg.get("Bcc", "") or "").strip()
        subject = str(msg.get("Subject", "") or "").strip()

        # Extract plain-text email body
        if msg.is_multipart():
            body_parts = []

            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    try:
                        body_parts.append(part.get_content())
                    except Exception:
                        pass

            body = "\n".join(body_parts)

        else:
            try:
                body = msg.get_content()
            except Exception:
                body = msg.get_payload()

    except Exception:
        # Fallback for malformed emails
        message_id = ""
        date = ""
        sender = ""
        recipient = ""
        cc = ""
        bcc = ""
        subject = ""

        parts = re.split(r"\n\s*\n", raw_message, maxsplit=1)
        body = parts[1] if len(parts) > 1 else raw_message

    return pd.Series({
        "message_id": message_id,
        "date": date,
        "from": sender,
        "to": recipient,
        "cc": cc,
        "bcc": bcc,
        "subject": subject,
        "body": body
    })


parsed = df["message"].apply(parse_enron_email)

df = pd.concat(
    [
        df[["file"]].reset_index(drop=True),
        parsed.reset_index(drop=True)
    ],
    axis=1
)


# =========================================================
# 3. NORMALIZE SUBJECT AND BODY
# =========================================================

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove null characters
    text = text.replace("\x00", " ")

    # Normalize all whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["subject"] = df["subject"].map(normalize_text)
df["body"] = df["body"].map(normalize_text)


# =========================================================
# 4. CREATE MODEL TEXT
# =========================================================
# Only Subject + Body are used as model input.
# Metadata headers are excluded from the classification text.

df["text"] = (
    df["subject"].fillna("") +
    " " +
    df["body"].fillna("")
).map(normalize_text)


# Remove emails with no usable textual content
before_empty = len(df)

df = df[df["text"].str.len() > 0].copy()

print("Rows after removing empty emails:", len(df))
print("Empty emails removed:", before_empty - len(df))


# =========================================================
# 5. CONTENT-BASED DEDUPLICATION
# =========================================================
# Message-ID is NOT used for deduplication because different
# Message-IDs may contain identical Subject + Body content.
#
# This prevents identical email content from appearing in
# both development and evaluation sets.

before_dedup = len(df)

unique_message_ids = df["message_id"].nunique()
unique_texts = df["text"].nunique()

duplicate_rows = df.duplicated(
    subset=["text"],
    keep="first"
).sum()

print("\nBefore content deduplication:", before_dedup)
print("Unique Message-IDs:", unique_message_ids)
print("Unique normalized texts:", unique_texts)
print("Duplicate content copies removable:", duplicate_rows)


# Keep one representative copy of each unique Subject + Body
df = (
    df.drop_duplicates(
        subset=["text"],
        keep="first"
    )
    .reset_index(drop=True)
)


after_dedup = len(df)

print("\nAfter content deduplication:", after_dedup)
print("Duplicate content copies removed:", before_dedup - after_dedup)


# Confirm content is now unique
assert df["text"].is_unique, \
    "Duplicate normalized email content remains."


# =========================================================
# 6. CREATE STABLE CONTENT-BASED DOCUMENT ID
# =========================================================
# Since classification is based on Subject + Body, the stable
# document identifier is generated from the same normalized
# textual content.

def create_doc_id(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()[:20]


df["doc_id"] = df["text"].map(create_doc_id)


assert df["doc_id"].is_unique, \
    "Duplicate document IDs detected after content deduplication."


# =========================================================
# 7. FINAL INTEGRITY CHECKS
# =========================================================

assert len(df) == df["text"].nunique()
assert len(df) == df["doc_id"].nunique()

print("\n" + "=" * 60)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 60)

print(f"Original records:             {before_empty:,}")
print(f"Empty records removed:        {before_empty - before_dedup:,}")
print(f"Duplicate content removed:    {before_dedup - after_dedup:,}")
print(f"Final unique email corpus:    {after_dedup:,}")

print("\nExtracted columns:")
print(df.columns.tolist())


# =========================================================
# 8. INSPECT SAMPLE
# =========================================================

display(
    df[
        [
            "file",
            "message_id",
            "date",
            "from",
            "to",
            "subject",
            "body",
            "doc_id"
        ]
    ].head()
)

In [ ]:
print("Unique Message-IDs:", df["message_id"].nunique())
print("Unique normalized texts:", df["text"].nunique())

duplicate_text_count = df.duplicated(subset=["text"], keep=False).sum()
print("Rows belonging to duplicate-text groups:", duplicate_text_count)

exact_duplicates_to_remove = df.duplicated(
    subset=["text"],
    keep="first"
).sum()

print("Exact content duplicates removable:", exact_duplicates_to_remove)

## 3. Fixed development / holdout split

The split is made **before keyword labeling**. Once generated, IDs are saved to Drive and reused in future runs.

In [ ]:
DEV_PATH = os.path.join(SPLIT_DIR, "development_ids.csv")
HOLDOUT_PATH = os.path.join(SPLIT_DIR, "holdout_ids.csv")
TEST_IDS_PATH = os.path.join(SPLIT_DIR, "human_test_ids.csv")

if all(os.path.exists(p) for p in [DEV_PATH, HOLDOUT_PATH, TEST_IDS_PATH]):
    dev_ids = pd.read_csv(DEV_PATH)["doc_id"].astype(str)
    holdout_ids = pd.read_csv(HOLDOUT_PATH)["doc_id"].astype(str)
    test_ids = pd.read_csv(TEST_IDS_PATH)["doc_id"].astype(str)
    print("Loaded existing locked split.")
else:
    development, holdout = train_test_split(
        df, test_size=HOLDOUT_FRACTION, random_state=SEED, shuffle=True
    )
    human_test = holdout.sample(
        n=min(N_HUMAN_TEST, len(holdout)), random_state=SEED
    )

    pd.DataFrame({"doc_id":development.doc_id}).to_csv(DEV_PATH,index=False)
    pd.DataFrame({"doc_id":holdout.doc_id}).to_csv(HOLDOUT_PATH,index=False)
    pd.DataFrame({"doc_id":human_test.doc_id}).to_csv(TEST_IDS_PATH,index=False)

    dev_ids = development.doc_id
    holdout_ids = holdout.doc_id
    test_ids = human_test.doc_id
    print("Created and saved new locked split.")

dev_df = df[df.doc_id.isin(set(dev_ids))].copy()
holdout_df = df[df.doc_id.isin(set(holdout_ids))].copy()
human_test_unlabeled = df[df.doc_id.isin(set(test_ids))].copy()

assert set(dev_df.doc_id).isdisjoint(set(human_test_unlabeled.doc_id))

print("Development:", len(dev_df))
print("Holdout:", len(holdout_df))
print("Manual test sample:", len(human_test_unlabeled))

## 4. Final predefined keyword vocabulary

The vocabulary is organized around the predefined organizational-operational relevance categories.

**Important methodological points**
- The vocabulary must be frozen before the manually labeled test set is evaluated.
- The final paper should state the actual provenance used to motivate these categories/terms.
- Development-corpus frequency is used only to describe/verify coverage, not to optimize against human test labels.
- Highly ambiguous/generic standalone terms from the old experiment such as `hr`, `spe`, `problem`, `issue`, `concern`, `numbers`, and `discussion` are **not part of this vocabulary**.

In [ ]:
KEYWORD_CATEGORIES = {
    "Energy and Trading": [
        "gas", "electricity", "power", "pipeline", "trading",
        "market", "pricing", "supply", "counterparty", "transaction"
    ],
    "Finance and Accounting": [
        "accounting", "audit", "credit", "debt", "derivatives",
        "earnings", "finance", "financials", "hedge", "hedging",
        "liability", "mark-to-market", "off-balance-sheet",
        "portfolio", "profit", "revenue", "risk", "valuation"
    ],
    "Contracts, Legal and Compliance": [
        "contract", "agreement", "legal", "compliance",
        "disclosure", "investigation", "policy",
    ],
    "Management and Governance": [
        "board", "executive", "management", "leadership",
        "approval", "strategy", "reorganization"
    ],
    "Human Resources and Personnel": [
        "employee", "personnel", "hire", "promotion", "bonus", "staff"
    ],
    "Organizational Activity": [
        "meeting", "agenda", "proposal", "report", "forecast"
    ]
}

KEYWORDS = sorted(set(
    term for terms in KEYWORD_CATEGORIES.values() for term in terms
))

print("Number of final keywords/phrases:", len(KEYWORDS))
print(KEYWORDS)

# Safety checks: these terms must never silently re-enter the labeling vocabulary.
FORBIDDEN_AMBIGUOUS_TERMS = {
    "hr", "spe", "problem", "issue", "concern", "numbers", "discussion"
}
assert not (FORBIDDEN_AMBIGUOUS_TERMS & set(KEYWORDS))

## 5. Save keyword methodology/provenance table

In [ ]:
rows = []
for category, terms in KEYWORD_CATEGORIES.items():
    for term in terms:
        rows.append({
            "keyword_or_phrase": term,
            "category": category,
            "selection_basis": "Predefined organizational-operational category",
            "source_reference": ""  # Fill with actual supporting source/citation if used.
        })

keyword_table = pd.DataFrame(rows).sort_values(["category","keyword_or_phrase"])
keyword_table.to_csv(
    os.path.join(OUT_DIR, "keyword_vocabulary_and_provenance.csv"), index=False
)
display(keyword_table)

## 6. Whole-word / complete-phrase keyword matching

A positive weak label is assigned when **at least one** predefined keyword or phrase appears.

Single terms are matched as whole lexical units; phrases are matched as complete phrases with flexible whitespace.

No stemming, lemmatization, translation, synonym expansion, or semantic expansion is used.

In [ ]:
def compile_keyword(term):
    escaped = re.escape(term).replace(r"\ ", r"\s+")
    return re.compile(rf"(?<!\w){escaped}(?!\w)", flags=re.IGNORECASE)

KEYWORD_PATTERNS = {k: compile_keyword(k) for k in KEYWORDS}

def get_keyword_matches(text):
    return [k for k,p in KEYWORD_PATTERNS.items() if p.search(text or "")]

def keyword_label(text):
    return int(bool(get_keyword_matches(text)))

dev_df["weak_label"] = dev_df["text"].map(keyword_label).astype(int)

print("Weak-label counts:")
print(dev_df["weak_label"].value_counts().sort_index())
print("Weak Relevant rate:", round(dev_df["weak_label"].mean(), 4))

## 7. Development-only keyword frequency and coverage diagnostics

In [ ]:
frequency_rows = []
for keyword, pattern in KEYWORD_PATTERNS.items():
    count = int(dev_df["text"].map(lambda x: bool(pattern.search(x))).sum())
    category = next(c for c,terms in KEYWORD_CATEGORIES.items() if keyword in terms)
    frequency_rows.append({
        "keyword_or_phrase": keyword,
        "category": category,
        "document_frequency": count,
        "document_percent": 100 * count / len(dev_df)
    })

keyword_frequency = pd.DataFrame(frequency_rows).sort_values(
    "document_frequency", ascending=False
)
keyword_frequency.to_csv(
    os.path.join(OUT_DIR, "keyword_frequency_development.csv"), index=False
)

coverage = pd.DataFrame([{
    "development_documents": len(dev_df),
    "weak_relevant": int(dev_df.weak_label.sum()),
    "weak_non_relevant": int((dev_df.weak_label == 0).sum()),
    "weak_relevant_rate": float(dev_df.weak_label.mean())
}])
coverage.to_csv(
    os.path.join(OUT_DIR, "weak_label_coverage_development.csv"), index=False
)

display(coverage)
display(keyword_frequency)

## 8. Export the ONE blinded manual test set

The file contains **no keyword matches and no weak labels**.

One author can label all approximately 1,000 emails. The absence of multiple annotators/inter-annotator agreement should then be stated transparently as a limitation.

In [ ]:
ANNOTATION_PATH = os.path.join(
    ANNOTATION_DIR,
    "human_test_annotation.csv"
)

# Columns useful for manual review
annotation_cols = [
    "doc_id",
    "date",
    "from",
    "to",
    "cc",
    "bcc",
    "subject",
    "body"
]

# Confirm all required columns exist
missing_cols = [
    c for c in annotation_cols
    if c not in human_test_unlabeled.columns
]

if missing_cols:
    raise ValueError(
        f"Missing annotation columns: {missing_cols}"
    )


# ---------------------------------------------------------
# Create blinded manual annotation file
# ---------------------------------------------------------

if not os.path.exists(ANNOTATION_PATH):

    annotation = (
        human_test_unlabeled[annotation_cols]
        .sample(frac=1, random_state=SEED)
        .copy()
    )

    # Human annotation fields
    annotation["manual_label"] = ""
    annotation["uncertain"] = ""
    annotation["notes"] = ""

    annotation.to_csv(
        ANNOTATION_PATH,
        index=False
    )

    print("Created annotation file:")
    print(ANNOTATION_PATH)

else:
    print("Annotation file already exists:")
    print(ANNOTATION_PATH)


print("\nNumber of emails for manual annotation:",
      len(human_test_unlabeled))

print("\nManual labeling:")
print("1 = Relevant")
print("0 = Non-Relevant")
print("uncertain = optional flag for difficult cases")

print(
    "\nIMPORTANT: Complete the manual annotation "
    "without viewing keyword predictions or model predictions."
)

## 9. Development train/validation split

The validation set remains weakly labeled. Therefore, validation performance measures how well a model reproduces the **weak labeling function**, not true human relevance. This distinction must be explicit in the paper.

In [ ]:
train_ids, val_ids = train_test_split(
    dev_df["doc_id"], test_size=0.20, random_state=SEED, shuffle=True
)

train_df = dev_df[dev_df.doc_id.isin(set(train_ids))].copy()
val_df = dev_df[dev_df.doc_id.isin(set(val_ids))].copy()

if MAX_TRAIN_ROWS is not None and len(train_df) > MAX_TRAIN_ROWS:
    train_df = train_df.sample(MAX_TRAIN_ROWS, random_state=SEED)

print("Weak-label train:", len(train_df))
print("Weak-label validation:", len(val_df))

## 10. Metrics and bootstrap confidence intervals

In [ ]:
def metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true,y_pred),
        "precision": precision_score(y_true,y_pred,zero_division=0),
        "recall": recall_score(y_true,y_pred,zero_division=0),
        "f1": f1_score(y_true,y_pred,zero_division=0)
    }

def bootstrap_ci(y_true, y_pred, fn, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred)
    n=len(y_true); vals=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,n)
        vals.append(fn(y_true[idx],y_pred[idx]))
    return np.percentile(vals,[2.5,97.5])

def save_confusion(y_true,y_pred,title,filename):
    cm=confusion_matrix(y_true,y_pred,labels=[0,1])
    fig,ax=plt.subplots(figsize=(5,5))
    ConfusionMatrixDisplay(
        cm,display_labels=["Non-Relevant","Relevant"]
    ).plot(ax=ax,colorbar=False,values_format="d")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(filename,dpi=200)
    plt.close(fig)

## 11. Logistic Regression and Linear SVM

In [ ]:
tfidf = TfidfVectorizer(min_df=3,max_df=0.9,ngram_range=(1,2))
X_train = tfidf.fit_transform(train_df["text"])
X_val = tfidf.transform(val_df["text"])
y_train = train_df["weak_label"].values
y_val = val_df["weak_label"].values

lr = LogisticRegression(max_iter=2000,random_state=SEED)
svm = LinearSVC(random_state=SEED)

lr.fit(X_train,y_train)
svm.fit(X_train,y_train)

weak_validation_results=[]
for name,model in [("Logistic Regression",lr),("Linear SVM",svm)]:
    pred=model.predict(X_val)
    weak_validation_results.append({"model":name,**metrics(y_val,pred)})

display(pd.DataFrame(weak_validation_results))

## 12. TextCNN

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Embedding,Conv1D,GlobalMaxPooling1D,Dense,Dropout

tf.random.set_seed(SEED)
MAX_WORDS=20000
MAX_LEN=200

cnn_tokenizer=Tokenizer(num_words=MAX_WORDS)
cnn_tokenizer.fit_on_texts(train_df["text"])

Xtr_cnn=pad_sequences(cnn_tokenizer.texts_to_sequences(train_df["text"]),maxlen=MAX_LEN)
Xva_cnn=pad_sequences(cnn_tokenizer.texts_to_sequences(val_df["text"]),maxlen=MAX_LEN)

inp=Input(shape=(MAX_LEN,))
x=Embedding(MAX_WORDS,128)(inp)
x=Conv1D(128,5,activation="relu")(x)
x=GlobalMaxPooling1D()(x)
x=Dense(64,activation="relu")(x)
x=Dropout(0.5)(x)
out=Dense(1,activation="sigmoid")(x)
cnn=Model(inp,out)
cnn.compile(loss="binary_crossentropy",optimizer="adam",metrics=["accuracy"])

cnn.fit(
    Xtr_cnn,y_train,
    validation_data=(Xva_cnn,y_val),
    epochs=6,batch_size=32,verbose=1
)

cnn_val_pred=(cnn.predict(Xva_cnn,verbose=0).ravel()>=0.5).astype(int)
weak_validation_results.append({"model":"TextCNN",**metrics(y_val,cnn_val_pred)})

## 13. DistilBERT

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,AutoModelForSequenceClassification,
    TrainingArguments,Trainer
)

MODEL_NAME="distilbert-base-uncased"
bert_tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return bert_tokenizer(
        batch["text"],truncation=True,padding="max_length",max_length=256
    )

train_hf=Dataset.from_pandas(
    train_df[["text","weak_label"]].rename(columns={"weak_label":"labels"}).reset_index(drop=True)
)
val_hf=Dataset.from_pandas(
    val_df[["text","weak_label"]].rename(columns={"weak_label":"labels"}).reset_index(drop=True)
)

train_hf=train_hf.map(tokenize_batch,batched=True,remove_columns=["text"])
val_hf=val_hf.map(tokenize_batch,batched=True,remove_columns=["text"])
train_hf.set_format("torch"); val_hf.set_format("torch")

bert_model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=2)

args=TrainingArguments(
    output_dir=os.path.join(OUT_DIR,"distilbert"),
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    save_strategy="no",
    report_to="none",
    seed=SEED
)

bert_trainer=Trainer(
    model=bert_model,args=args,train_dataset=train_hf,eval_dataset=val_hf
)
bert_trainer.train()

bert_val_pred=np.argmax(bert_trainer.predict(val_hf).predictions,axis=1)
weak_validation_results.append(
    {"model":"DistilBERT",**metrics(y_val,bert_val_pred)}
)

weak_val_table=pd.DataFrame(weak_validation_results)
weak_val_table.to_csv(
    os.path.join(OUT_DIR,"weak_label_validation_results.csv"),index=False
)
display(weak_val_table)

# FINAL HUMAN-TEST BOUNDARY

Do not continue until:
- the approximately 1,000 emails have been manually labeled;
- the keyword vocabulary and matching rule are frozen;
- model configurations are frozen;
- `FINAL_EVALUATION` is changed to `True`.

The cells below inspect the human labels for the first time.

## 14. Load and validate manual labels

In [ ]:
if not FINAL_EVALUATION:
    raise RuntimeError(
        "FINAL_EVALUATION=False. Complete annotation and freeze development choices first."
    )

ann=pd.read_csv(ANNOTATION_PATH)
ann["manual_label"]=ann["manual_label"].astype(str).str.strip()

bad=ann[~ann.manual_label.isin(["0","1"])]
if len(bad):
    raise ValueError(f"{len(bad)} annotation rows are missing a valid 0/1 manual label.")

ann["manual_label"]=ann["manual_label"].astype(int)

test_df=human_test_unlabeled.merge(
    ann[["doc_id","manual_label"]],on="doc_id",how="inner"
)
assert len(test_df)==len(human_test_unlabeled)

print("Human test size:",len(test_df))
print(test_df.manual_label.value_counts().sort_index())
print("Human Relevant rate:",test_df.manual_label.mean())

## 15. Keyword rule = explicit baseline

The exact rule that generated the weak supervision is evaluated directly against the independent human labels. This is the main baseline requested by the reviewers.

In [ ]:
y_test=test_df["manual_label"].values
test_df["keyword_prediction"]=test_df["text"].map(keyword_label).astype(int)

FINAL_RESULTS=[]
PREDICTIONS={}

def add_final_result(name,pred):
    pred=np.asarray(pred)
    m=metrics(y_test,pred)
    acc_ci=bootstrap_ci(y_test,pred,accuracy_score)
    f1_ci=bootstrap_ci(
        y_test,pred,lambda a,b:f1_score(a,b,zero_division=0)
    )
    FINAL_RESULTS.append({
        "method":name,**m,
        "accuracy_ci_low":acc_ci[0],"accuracy_ci_high":acc_ci[1],
        "f1_ci_low":f1_ci[0],"f1_ci_high":f1_ci[1]
    })
    PREDICTIONS[name]=pred
    save_confusion(
        y_test,pred,name,
        os.path.join(OUT_DIR,"cm_"+re.sub(r"[^a-z0-9]+","_",name.lower()).strip("_")+".png")
    )

add_final_result("Keyword rule baseline",test_df["keyword_prediction"].values)
display(pd.DataFrame(FINAL_RESULTS))

## 16. Logistic Regression and SVM on human test

In [ ]:
X_test=tfidf.transform(test_df["text"])

add_final_result("Logistic Regression",lr.predict(X_test))
add_final_result("Linear SVM",svm.predict(X_test))

## 17. TextCNN on human test

In [ ]:
Xtest_cnn=pad_sequences(
    cnn_tokenizer.texts_to_sequences(test_df["text"]),maxlen=MAX_LEN
)
cnn_test_pred=(cnn.predict(Xtest_cnn,verbose=0).ravel()>=0.5).astype(int)
add_final_result("TextCNN",cnn_test_pred)

## 18. DistilBERT on human test

In [ ]:
test_hf=Dataset.from_pandas(test_df[["text"]].reset_index(drop=True))
test_hf=test_hf.map(tokenize_batch,batched=True,remove_columns=["text"])
test_hf.set_format("torch")

bert_test_pred=np.argmax(bert_trainer.predict(test_hf).predictions,axis=1)
add_final_result("DistilBERT",bert_test_pred)

## 19. Final camera-ready results

In [ ]:
final_table=pd.DataFrame(FINAL_RESULTS)
final_table.to_csv(
    os.path.join(OUT_DIR,"final_human_test_results.csv"),index=False
)

prediction_table=pd.DataFrame({
    "doc_id":test_df.doc_id,
    "manual_label":y_test
})
for name,pred in PREDICTIONS.items():
    prediction_table[name]=pred
prediction_table.to_csv(
    os.path.join(OUT_DIR,"final_test_predictions.csv"),index=False
)

display(final_table.sort_values("f1",ascending=False))

In [ ]:
# =========================================================
# CONFUSION MATRICES — FINAL HUMAN-LABELED TEST SET
# =========================================================

import os
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# ---------------------------------------------------------
# Predictions from the final evaluation
# ---------------------------------------------------------

models_predictions = {
    "Keyword Rule": test_df["keyword_prediction"].values,
    "Logistic Regression": lr.predict(X_test),
    "Linear SVM": svm.predict(X_test),
    "TextCNN": cnn_test_pred,
    "DistilBERT": bert_test_pred
}

# Ground-truth human labels
y_true = test_df["manual_label"].values


# ---------------------------------------------------------
# Generate one confusion matrix per model
# ---------------------------------------------------------

for model_name, y_pred in models_predictions.items():

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    # Matrix structure:
    #
    #               Predicted
    #               NR      R
    # Actual NR     TN      FP
    # Actual R      FN      TP

    tn, fp, fn, tp = cm.ravel()

    # Percentage relative to entire test set
    cm_percent = (cm / cm.sum()) * 100


    # -----------------------------------------------------
    # Plot
    # -----------------------------------------------------

    fig, ax = plt.subplots(figsize=(6, 5))

    im = ax.imshow(cm, cmap="Blues")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels([
        "Non-Relevant",
        "Relevant"
    ])

    ax.set_yticklabels([
        "Non-Relevant",
        "Relevant"
    ])

    ax.set_xlabel(
        "Predicted Label",
        fontsize=11
    )

    ax.set_ylabel(
        "Human Label",
        fontsize=11
    )

    ax.set_title(
        f"{model_name}",
        fontsize=13,
        fontweight="bold"
    )


    # -----------------------------------------------------
    # Add counts + percentages inside cells
    # -----------------------------------------------------

    threshold = cm.max() / 2

    for i in range(2):
        for j in range(2):

            ax.text(
                j,
                i,
                f"{cm[i, j]:,}\n"
                f"({cm_percent[i, j]:.1f}%)",
                ha="center",
                va="center",
                color="white" if cm[i, j] > threshold else "black",
                fontsize=12,
                fontweight="bold"
            )


    # -----------------------------------------------------
    # Add TN / FP / FN / TP labels
    # -----------------------------------------------------

    cell_labels = [
        ["TN", "FP"],
        ["FN", "TP"]
    ]

    for i in range(2):
        for j in range(2):

            ax.text(
                j,
                i + 0.27,
                cell_labels[i][j],
                ha="center",
                va="center",
                fontsize=9,
                color="white" if cm[i, j] > threshold else "black"
            )


    plt.colorbar(im, ax=ax)

    plt.tight_layout()


    # -----------------------------------------------------
    # Save figure
    # -----------------------------------------------------

    safe_name = re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        model_name
    ).strip("_").lower()

    save_path = os.path.join(
        OUT_DIR,
        f"confusion_matrix_{safe_name}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


    # -----------------------------------------------------
    # Print numerical values
    # -----------------------------------------------------

    print(f"\n{model_name}")
    print("-" * 50)

    print(f"True Negatives  (TN): {tn}")
    print(f"False Positives (FP): {fp}")
    print(f"False Negatives (FN): {fn}")
    print(f"True Positives  (TP): {tp}")

    print(f"Saved to: {save_path}")

## 20. Reproducibility manifest

In [ ]:
manifest={
    "seed":SEED,
    "clean_unique_emails":int(len(df)),
    "development_documents":int(len(dev_df)),
    "holdout_documents":int(len(holdout_df)),
    "human_test_documents":int(len(test_df)),
    "keyword_count":int(len(KEYWORDS)),
    "keyword_rule":"Relevant if at least one predefined keyword/phrase is matched as a whole word/phrase",
    "stemming":False,
    "lemmatization":False,
    "translation":False,
    "semantic_expansion":False,
    "models":["Logistic Regression","Linear SVM","TextCNN","DistilBERT"],
    "human_test_used_for_development":False
}
with open(os.path.join(OUT_DIR,"experiment_manifest.json"),"w") as f:
    json.dump(manifest,f,indent=2)

print("Complete. Results:",OUT_DIR)